# Fitting ESR Functions to Galaxy and Halo Mass Functions

This notebook provides code to fit the best ESR (Exhaustive Symbolic Regression) functions
from [Ford, Desmond, Bartlett & Ferreira (2026)](https://arxiv.org/abs/2604.23236) to luminosity functions (LF),
stellar mass functions (SMF), and halo mass functions (HMF).

By default it uses the datasets from the paper (included in the `data/` directory).
To use your own data, see the **"Using your own data"** section below.

**Requirements:** `numpy`, `scipy`, `matplotlib`

## Using your own data

To fit these functions to your own LF, SMF or HMF:

1. **LF / SMF:** Prepare a file with columns `[x, log10(phi), sigma_log10phi, Veff]`,
   where `x = L / (1e9 L_sun)` or `x = M* / (1e9 M_sun)`, `phi` is the number density
   in Mpc$^{-3}$ dex$^{-1}$, and `Veff` is the effective survey volume per bin in Mpc$^3$.
   Then replace the file path in the `load_lf_smf()` call in the relevant section below.

2. **HMF:** Prepare a file with columns `[sigma, counts, |dlnsigma/dlogM|, norm, log10(M)]`,
   where `norm = |dlnsigma/dlogM| * (rho_m / M) * Veff * dlogM` converts $f(\sigma)$ to
   predicted counts. Replace the file path in the HMF loading cell.
   Alternatively, if you have $\sigma$ values and halo counts with a known volume,
   you can compute `norm` from the cosmology using e.g. the
   [`hmf`](https://github.com/halomod/hmf) Python package.

3. Set `FIT_FROM_SCRATCH = True` to fit the ESR functions to your data,
   or `False` to evaluate them at the paper's best-fit parameters.

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

## Helper: Poisson log-likelihood and fitting routine

All fits use a Poisson likelihood on binned counts, as in the paper.

In [ ]:
def poisson_nll(params, x, counts, model_func, pred_to_counts):
    """Negative Poisson log-likelihood (ignoring constant ln(N!) term).

    Args:
        params: model parameters
        x: input variable (L/1e9 L_sun, M*/1e9 M_sun, or sigma)
        counts: observed counts per bin
        model_func: function(x, *params) -> phi or f(sigma)
        pred_to_counts: function(phi_or_f, x) -> predicted counts lambda
    """
    try:
        pred = model_func(x, *params)
        lam = pred_to_counts(pred, x)
        if np.any(lam <= 0) or np.any(~np.isfinite(lam)):
            return 1e30
        return np.sum(lam - counts * np.log(lam))
    except (ValueError, FloatingPointError, OverflowError):
        return 1e30


def fit_model(model_func, p0, x, counts, pred_to_counts, n_restarts=5):
    """Fit model_func to data, returning best-fit parameters and NLL."""
    best_res = None
    best_nll = np.inf
    for i in range(n_restarts):
        if i == 0:
            p_init = np.array(p0, dtype=float)
        else:
            p_init = np.array(p0, dtype=float) * (1 + 0.3 * np.random.randn(len(p0)))
        res = minimize(poisson_nll, p_init,
                       args=(x, counts, model_func, pred_to_counts),
                       method='Nelder-Mead',
                       options={'maxiter': 50000, 'xatol': 1e-10, 'fatol': 1e-10})
        if res.fun < best_nll:
            best_nll = res.fun
            best_res = res
    return best_res.x, best_res.fun


def plot_fit(x_data, y_data, y_err, x_plot, y_plot, xlabel, ylabel,
             func_label, title):
    """Plot data with error bars and fitted function."""
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(x_data, y_data, yerr=y_err, fmt='x', color='black',
                label='Data', zorder=5)
    ax.plot(x_plot, y_plot, color='C0', label=func_label, zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 1. Luminosity Function (LF)

### Best ESR functions

**Sersic photometry** (Eq. 11, rank 2 in Table 1):
$$\phi(x) = e^{\alpha\,(\ln(\beta + \gamma^{\,x}))^{\delta}}$$

**cmodel photometry** (Eq. 12, rank 1 in Table 1):
$$\phi(x) = (\alpha + |\beta|^{x^\gamma})^\delta$$

where $x = L \,/\, 10^9\,L_\odot$ in both cases.

In [ ]:
# === LF: Model definitions ===

def lf_sersic_esr(x, alpha, beta, gamma, delta):
    """Eq. 11: phi(x) = exp(alpha * (ln(beta + gamma^x))^delta)"""
    return np.exp(alpha * np.log(beta + np.power(gamma, x))**delta)

def lf_cmodel_esr(x, alpha, beta, gamma, delta):
    """Eq. 12: phi(x) = |alpha + |beta|^{x^gamma}|^delta"""
    return np.abs(alpha + np.abs(beta)**np.power(x, gamma))**delta

# Best-fit parameters from Table 1 (paper notation -> Eq. notation)
# LF Sersic (rank 2): theta = [-2.33e-3, 0.985, 1.028, 0.302]
#   alpha = ln|theta_0|, beta = theta_1, gamma = theta_2, delta = theta_3
LF_SERSIC_PARAMS = dict(
    alpha=np.log(2.33e-3), beta=0.985, gamma=1.028, delta=0.302)

# LF cmodel (rank 1): theta = [17.0, -1.80, 0.455, -1.85]
#   alpha = theta_0, beta = theta_1, gamma = theta_2, delta = theta_3
LF_CMODEL_PARAMS = dict(
    alpha=17.0, beta=-1.80, gamma=0.455, delta=-1.85)

In [ ]:
# === LF: Load data ===
# Replace these paths with your own data.
# Expected format: columns [x, log10(phi), sigma_log10phi, Veff]
#   x = L / (1e9 L_sun)
#   phi in units of Mpc^-3 dex^-1
#   Veff = effective volume per bin in Mpc^3

def load_lf_smf(filepath):
    """Load LF or SMF data and return x, counts, Veff, log10(phi), sigma."""
    x, log10phi, sigma_log10phi, Veff = np.loadtxt(
        filepath, dtype=float, unpack=True)
    delta_logx = np.log10(x[1]) - np.log10(x[0])  # bin width in dex
    counts = 10**log10phi * Veff * delta_logx
    return x, counts, Veff, delta_logx, log10phi, sigma_log10phi

lf_ser_x, lf_ser_counts, lf_ser_Veff, lf_ser_dlogx, lf_ser_y, lf_ser_yerr = \
    load_lf_smf('data/LF_Ser_L.txt')

lf_cm_x, lf_cm_counts, lf_cm_Veff, lf_cm_dlogx, lf_cm_y, lf_cm_yerr = \
    load_lf_smf('data/LF_cmodel.txt')

print(f'LF Sersic:  {len(lf_ser_x)} bins, x = {lf_ser_x[0]:.2f} -- {lf_ser_x[-1]:.2f}')
print(f'LF cmodel:  {len(lf_cm_x)} bins, x = {lf_cm_x[0]:.2f} -- {lf_cm_x[-1]:.2f}')

In [ ]:
# === LF: Fit or use paper parameters ===

FIT_FROM_SCRATCH = True  # Set False to use the paper's best-fit values

def lf_pred_to_counts(phi, x, Veff, dlogx):
    return phi * Veff * dlogx

if FIT_FROM_SCRATCH:
    p0_ser = list(LF_SERSIC_PARAMS.values())
    lf_ser_pfit, lf_ser_nll = fit_model(
        lf_sersic_esr, p0_ser, lf_ser_x, lf_ser_counts,
        lambda phi, x: lf_pred_to_counts(phi, x, lf_ser_Veff, lf_ser_dlogx))
    print(f'LF Sersic fit:  alpha={lf_ser_pfit[0]:.4f}, beta={lf_ser_pfit[1]:.4f}, '
          f'gamma={lf_ser_pfit[2]:.4f}, delta={lf_ser_pfit[3]:.4f}  (NLL={lf_ser_nll:.1f})')

    p0_cm = list(LF_CMODEL_PARAMS.values())
    lf_cm_pfit, lf_cm_nll = fit_model(
        lf_cmodel_esr, p0_cm, lf_cm_x, lf_cm_counts,
        lambda phi, x: lf_pred_to_counts(phi, x, lf_cm_Veff, lf_cm_dlogx))
    print(f'LF cmodel fit:  alpha={lf_cm_pfit[0]:.4f}, beta={lf_cm_pfit[1]:.4f}, '
          f'gamma={lf_cm_pfit[2]:.4f}, delta={lf_cm_pfit[3]:.4f}  (NLL={lf_cm_nll:.1f})')
else:
    lf_ser_pfit = list(LF_SERSIC_PARAMS.values())
    lf_cm_pfit = list(LF_CMODEL_PARAMS.values())
    print('Using paper best-fit parameters.')

In [ ]:
# === LF: Plot ===

for label, x, y, yerr, pfit, func in [
        ('Sersic', lf_ser_x, lf_ser_y, lf_ser_yerr,
         lf_ser_pfit, lf_sersic_esr),
        ('cmodel', lf_cm_x, lf_cm_y, lf_cm_yerr,
         lf_cm_pfit, lf_cmodel_esr)]:
    logL = np.log10(x) + 9  # log10(L / L_sun)
    x_fine = np.logspace(np.log10(x.min()) - 0.3,
                         np.log10(x.max()) + 0.5, 300)
    y_fine = np.log10(func(x_fine, *pfit))
    logL_fine = np.log10(x_fine) + 9
    plot_fit(logL, y, yerr, logL_fine, y_fine,
             r'$\log_{10}(L\,/\,L_\odot)$',
             r'$\log_{10}(\phi\;/\;\mathrm{Mpc}^{-3}\,\mathrm{dex}^{-1})$',
             f'ESR best ({label})',
             f'Luminosity Function — {label}')

---
## 2. Stellar Mass Function (SMF)

### Best ESR functions

**Sersic photometry** (rank 2 in Table 2):
$$\phi(x) = |\ln|\theta_0| + |\theta_1|^{x^{\theta_2}}|^{\theta_3}$$

**cmodel photometry** (Eq. 12, rank 1 in Table 2 — same form as LF cmodel):
$$\phi(x) = (\alpha + |\beta|^{x^\gamma})^\delta$$

where $x = M_\star \,/\, 10^9\,M_\odot$.

In [ ]:
# === SMF: Model definitions ===

def smf_sersic_esr(x, t0, t1, t2, t3):
    """Rank 2, Table 2: |ln|t0| + |t1|^{x^t2}|^t3"""
    return np.abs(np.log(np.abs(t0)) + np.abs(t1)**np.power(x, t2))**t3

smf_cmodel_esr = lf_cmodel_esr  # Same functional form (Eq. 12)

# Best-fit parameters from Table 2
SMF_SERSIC_PARAMS = dict(t0=2.61e21, t1=3.05, t2=0.299, t3=-1.22)
SMF_CMODEL_PARAMS = dict(alpha=26.9, beta=1.80, gamma=0.402, delta=-1.44)

In [ ]:
# === SMF: Load data ===
# Same format as LF: [x, log10(phi), sigma_log10phi, Veff]
#   x = M* / (1e9 M_sun)

smf_ser_x, smf_ser_counts, smf_ser_Veff, smf_ser_dlogx, smf_ser_y, smf_ser_yerr = \
    load_lf_smf('data/SMF_Ser_M.txt')

smf_cm_x, smf_cm_counts, smf_cm_Veff, smf_cm_dlogx, smf_cm_y, smf_cm_yerr = \
    load_lf_smf('data/SMF_cmodel_M.txt')

print(f'SMF Sersic:  {len(smf_ser_x)} bins, x = {smf_ser_x[0]:.2f} -- {smf_ser_x[-1]:.2f}')
print(f'SMF cmodel:  {len(smf_cm_x)} bins, x = {smf_cm_x[0]:.2f} -- {smf_cm_x[-1]:.2f}')

In [ ]:
# === SMF: Fit or use paper parameters ===

if FIT_FROM_SCRATCH:
    p0_ser = list(SMF_SERSIC_PARAMS.values())
    smf_ser_pfit, smf_ser_nll = fit_model(
        smf_sersic_esr, p0_ser, smf_ser_x, smf_ser_counts,
        lambda phi, x: lf_pred_to_counts(phi, x, smf_ser_Veff, smf_ser_dlogx))
    print(f'SMF Sersic fit:  t0={smf_ser_pfit[0]:.3e}, t1={smf_ser_pfit[1]:.4f}, '
          f't2={smf_ser_pfit[2]:.4f}, t3={smf_ser_pfit[3]:.4f}  (NLL={smf_ser_nll:.1f})')

    p0_cm = list(SMF_CMODEL_PARAMS.values())
    smf_cm_pfit, smf_cm_nll = fit_model(
        smf_cmodel_esr, p0_cm, smf_cm_x, smf_cm_counts,
        lambda phi, x: lf_pred_to_counts(phi, x, smf_cm_Veff, smf_cm_dlogx))
    print(f'SMF cmodel fit:  alpha={smf_cm_pfit[0]:.4f}, beta={smf_cm_pfit[1]:.4f}, '
          f'gamma={smf_cm_pfit[2]:.4f}, delta={smf_cm_pfit[3]:.4f}  (NLL={smf_cm_nll:.1f})')
else:
    smf_ser_pfit = list(SMF_SERSIC_PARAMS.values())
    smf_cm_pfit = list(SMF_CMODEL_PARAMS.values())
    print('Using paper best-fit parameters.')

In [ ]:
# === SMF: Plot ===

for label, x, y, yerr, pfit, func in [
        ('Sersic', smf_ser_x, smf_ser_y, smf_ser_yerr,
         smf_ser_pfit, smf_sersic_esr),
        ('cmodel', smf_cm_x, smf_cm_y, smf_cm_yerr,
         smf_cm_pfit, smf_cmodel_esr)]:
    logM = np.log10(x) + 9  # log10(M* / M_sun)
    x_fine = np.logspace(np.log10(x.min()) - 0.3,
                         np.log10(x.max()) + 0.5, 300)
    y_fine = np.log10(func(x_fine, *pfit))
    logM_fine = np.log10(x_fine) + 9
    plot_fit(logM, y, yerr, logM_fine, y_fine,
             r'$\log_{10}(M_\star\,/\,M_\odot)$',
             r'$\log_{10}(\phi\;/\;\mathrm{Mpc}^{-3}\,\mathrm{dex}^{-1})$',
             f'ESR best ({label})',
             f'Stellar Mass Function — {label}')

---
## 3. Halo Mass Function (HMF)

### Best ESR function (Eq. 13, rank 1 in Table 3)

$$f(\sigma) = \frac{\theta_0}{\theta_1 + e^{\sigma^{\theta_2 + \sigma}}}$$

The HMF is parametrised through the multiplicity function $f(\sigma)$:
$$n(M_h)\,\mathrm{d}\log M_h = f(\sigma)\,\frac{\bar{\rho}_m}{M_h}\,|\mathrm{d}\ln\sigma|$$

The input variable for ESR is $\sigma$ (the mass variance).

In [ ]:
# === HMF: Model definition ===

def hmf_esr(sigma, t0, t1, t2):
    """Eq. 13: f(sigma) = t0 / (t1 + exp(sigma^(t2 + sigma)))"""
    return t0 / (t1 + np.exp(np.power(sigma, t2 + sigma)))

# Best-fit parameters from Table 3 (realisation 50)
HMF_PARAMS = dict(t0=0.83, t1=0.48, t2=-2.22)

In [ ]:
# === HMF: Load data ===
# Default: Quijote realisation 0.
# HMF file format: [sigma, counts, |dlnsigma/dlogM|, norm, log10(M/(M_sun/h))]
#   norm = |dlnsigma/dlogM| * (rho_m/M) * Veff * dlogM
#   predicted counts = f(sigma) * norm
# mass_variance_multiplier.txt: [log10(M), sigma, factor]
#   factor = |dlnsigma/dlogM| * (rho_m/M) / Veff  (for converting f -> phi)

hmf_sim = 0  # Change to use a different realisation (0-99)

hmf_data = np.loadtxt(f'data/hmf_files/hmf_{hmf_sim}.dat', dtype=float)
hmf_sigma = hmf_data[:, 0]
hmf_counts = hmf_data[:, 1]
hmf_norm = hmf_data[:, 3]
hmf_logM = hmf_data[:, 4]

# Fiducial mass range: drop first 2 bins (< 50 particles per halo)
hmf_sigma = hmf_sigma[2:]
hmf_counts = hmf_counts[2:]
hmf_norm = hmf_norm[2:]
hmf_logM = hmf_logM[2:]

# For plotting: compute log10(phi) = log10(counts / (Veff * dlogM))
Veff_hmf = 1e9 / 0.6711**3  # (1 Gpc/h)^3 in Mpc^3
dlogM_hmf = 0.2
hmf_log10phi = np.log10(hmf_counts / (Veff_hmf * dlogM_hmf))
hmf_yerr = 1.0 / (np.log(10) * np.sqrt(hmf_counts))

print(f'HMF (sim {hmf_sim}): {len(hmf_sigma)} bins, '
      f'sigma = {hmf_sigma[-1]:.3f} -- {hmf_sigma[0]:.3f}')

In [ ]:
# === HMF: Fit or use paper parameters ===

def hmf_pred_to_counts(f_sigma, sigma):
    return f_sigma * hmf_norm

if FIT_FROM_SCRATCH:
    p0 = list(HMF_PARAMS.values())
    hmf_pfit, hmf_nll = fit_model(
        hmf_esr, p0, hmf_sigma, hmf_counts, hmf_pred_to_counts)
    print(f'HMF fit:  t0={hmf_pfit[0]:.4f}, t1={hmf_pfit[1]:.4f}, '
          f't2={hmf_pfit[2]:.4f}  (NLL={hmf_nll:.1f})')
else:
    hmf_pfit = list(HMF_PARAMS.values())
    print('Using paper best-fit parameters.')

In [ ]:
# === HMF: Plot ===

sigma_fine = np.linspace(hmf_sigma[-1] * 0.8, hmf_sigma[0] * 1.1, 300)
f_fine = hmf_esr(sigma_fine, *hmf_pfit)

fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(hmf_sigma, np.log10(hmf_counts / hmf_norm),
            yerr=hmf_yerr, fmt='x', color='black', label='Data', zorder=5)
ax.plot(sigma_fine, np.log10(f_fine), color='C0',
        label=r'ESR best: $\theta_0/(\theta_1 + e^{\sigma^{\theta_2+\sigma}})$')
ax.set_xlabel(r'$\sigma$')
ax.set_ylabel(r'$\log_{10}\,f(\sigma)$')
ax.set_title(f'Halo Mass Function — Quijote realisation {hmf_sim}')
ax.legend()
plt.tight_layout()
plt.show()